# JS05 - Langkah Kerja Praktik

Notebook ini berisi Praktikum 1 sampai 5 Jobsheet 5, serta pengujian tambahan untuk Penugasan 3 dan 4. Jalankan sel secara berurutan untuk melihat output dan visualisasi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, img_as_float, img_as_ubyte, color, transform
from scipy.spatial import distance
from skimage.metrics import structural_similarity as ssim
from skimage.util import random_noise
from skimage.filters import gaussian
from skimage.feature import match_template
from sklearn.cluster import KMeans
import warnings

plt.style.use('default')

def calculate_rgb_histogram(image, bins=16):
    image_uint8 = img_as_ubyte(np.clip(image, 0, 1))
    hist_r, _ = np.histogram(image_uint8[:, :, 0].ravel(), bins=bins, range=(0, 256))
    hist_g, _ = np.histogram(image_uint8[:, :, 1].ravel(), bins=bins, range=(0, 256))
    hist_b, _ = np.histogram(image_uint8[:, :, 2].ravel(), bins=bins, range=(0, 256))
    hist_combined = np.concatenate((hist_r, hist_g, hist_b)).astype(float)
    hist_sum = np.sum(hist_combined)
    if hist_sum > 0:
        hist_combined /= hist_sum
    return hist_combined

def calculate_mean_rgb(image):
    return image.reshape(-1, 3).mean(axis=0)

## Praktikum 1 — Menghitung Jarak Berbasis Piksel (Euclidean & Manhattan)

### Tujuan
Menghitung dan membandingkan jarak Euclidean (L2) dan Manhattan (L1) antara dua patch citra sederhana.

In [ ]:
image = img_as_float(data.camera())
patch1 = image[100:150, 100:150]
patch2 = image[100:150, 300:350]
patch3 = np.clip(patch1 + 0.1, 0, 1)

vec1 = patch1.flatten()
vec2 = patch2.flatten()
vec3 = patch3.flatten()

dist_l2_12 = distance.euclidean(vec1, vec2)
dist_l2_13 = distance.euclidean(vec1, vec3)
dist_l1_12 = distance.cityblock(vec1, vec2)
dist_l1_13 = distance.cityblock(vec1, vec3)

print(f'Jarak Euclidean antara Patch 1 dan Patch 2: {dist_l2_12:.4f}')
print(f'Jarak Euclidean antara Patch 1 dan Patch 3: {dist_l2_13:.4f}')
print(f'Jarak Manhattan antara Patch 1 dan Patch 2: {dist_l1_12:.4f}')
print(f'Jarak Manhattan antara Patch 1 dan Patch 3: {dist_l1_13:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(patch1, cmap='gray')
axes[0].set_title('Patch 1')
axes[0].axis('off')
axes[1].imshow(patch2, cmap='gray')
axes[1].set_title('Patch 2')
axes[1].axis('off')
axes[2].imshow(patch3, cmap='gray')
axes[2].set_title('Patch 3 (Mirip Patch 1)')
axes[2].axis('off')
plt.tight_layout()
plt.show()

### Hasil Praktikum 1
Patch yang lebih mirip memiliki jarak lebih kecil. Pada pengujian ini, Patch 1 dan Patch 3 jauh lebih dekat dibanding Patch 1 dan Patch 2. Tren jarak pada L1 dan L2 konsisten, meski skala nilainya berbeda.

## Praktikum 2 — Menghitung Cosine Similarity antara Histogram Warna

### Tujuan
Menghitung kemiripan dua citra berwarna berdasarkan histogram warna menggunakan Cosine Similarity.

In [ ]:
image1 = data.astronaut()
image2 = data.coffee()
image3 = data.astronaut()
image4 = image1[::2, ::2, :]

hist1 = calculate_rgb_histogram(image1)
hist2 = calculate_rgb_histogram(image2)
hist3 = calculate_rgb_histogram(image3)
hist4 = calculate_rgb_histogram(image4)

sim_12 = 1 - distance.cosine(hist1, hist2)
sim_13 = 1 - distance.cosine(hist1, hist3)
sim_14 = 1 - distance.cosine(hist1, hist4)

print(f'Cosine Similarity antara Image 1 (Astronaut) dan Image 2 (Coffee): {sim_12:.4f}')
print(f'Cosine Similarity antara Image 1 (Astronaut) dan Image 3 (Astronaut): {sim_13:.4f}')
print(f'Cosine Similarity antara Image 1 (Astronaut) dan Image 4 (Astronaut Downsampled): {sim_14:.4f}')

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(image1); axes[0].set_title('Image 1'); axes[0].axis('off')
axes[1].imshow(image2); axes[1].set_title('Image 2'); axes[1].axis('off')
axes[2].imshow(image3); axes[2].set_title('Image 3 (Same)'); axes[2].axis('off')
axes[3].imshow(image4); axes[3].set_title('Image 4 (Downsampled)'); axes[3].axis('off')
plt.tight_layout()
plt.show()

### Hasil Praktikum 2
Citra identik menghasilkan similarity 1.0. Citra dengan distribusi warna berbeda memiliki similarity lebih rendah, sedangkan citra asli dan versi downsampled tetap sangat mirip karena histogram warna globalnya hampir sama.

## Praktikum 3 — Menghitung Structural Similarity Index (SSIM)

### Tujuan
Mengukur kemiripan struktural antar citra menggunakan SSIM.

In [ ]:
image_ref = img_as_float(data.camera())
image_same = image_ref.copy()
image_noisy = random_noise(image_ref, mode='gaussian', var=0.01)
image_contrast = np.clip(image_ref * 0.8, 0, 1)
image_blurred = gaussian(image_ref, sigma=1.5, channel_axis=None)
data_range = image_ref.max() - image_ref.min()

ssim_same, diff_same = ssim(image_ref, image_same, data_range=data_range, full=True)
ssim_noisy, diff_noisy = ssim(image_ref, image_noisy, data_range=data_range, full=True)
ssim_contrast, diff_contrast = ssim(image_ref, image_contrast, data_range=data_range, full=True)
ssim_blurred, diff_blurred = ssim(image_ref, image_blurred, data_range=data_range, full=True)

print(f'SSIM (Ref vs Same): {ssim_same:.4f}')
print(f'SSIM (Ref vs Noisy): {ssim_noisy:.4f}')
print(f'SSIM (Ref vs Contrast): {ssim_contrast:.4f}')
print(f'SSIM (Ref vs Blurred): {ssim_blurred:.4f}')

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
ax = axes.ravel()
ax[0].imshow(image_ref, cmap='gray'); ax[0].set_title('Reference'); ax[0].axis('off')
ax[1].imshow(image_noisy, cmap='gray'); ax[1].set_title(f'Noisy\nSSIM: {ssim_noisy:.3f}'); ax[1].axis('off')
ax[2].imshow(image_contrast, cmap='gray'); ax[2].set_title(f'Contrast\nSSIM: {ssim_contrast:.3f}'); ax[2].axis('off')
ax[3].imshow(image_blurred, cmap='gray'); ax[3].set_title(f'Blurred\nSSIM: {ssim_blurred:.3f}'); ax[3].axis('off')
ax[4].imshow(np.zeros_like(image_ref), cmap='gray'); ax[4].set_title(''); ax[4].axis('off')
ax[5].imshow(diff_noisy, cmap='viridis'); ax[5].set_title('SSIM Difference (Noisy)'); ax[5].axis('off')
ax[6].imshow(diff_contrast, cmap='viridis'); ax[6].set_title('SSIM Difference (Contrast)'); ax[6].axis('off')
ax[7].imshow(diff_blurred, cmap='viridis'); ax[7].set_title('SSIM Difference (Blurred)'); ax[7].axis('off')
plt.tight_layout()
plt.show()

### Hasil Praktikum 3
SSIM bernilai 1.0 untuk citra yang sama. Noise menurunkan SSIM paling besar, blur menurunkannya secara sedang, dan perubahan kontras masih mempertahankan struktur utama sehingga SSIM tetap tinggi.

## Praktikum 4 — Penerapan Template Matching

### Tujuan
Menemukan lokasi objek kecil (template) di dalam citra yang lebih besar menggunakan normalized cross-correlation.

In [ ]:
image = data.coins()
template = image[15:80, 190:260]
result = match_template(image, template)
ij = np.unravel_index(np.argmax(result), result.shape)
x, y = ij[::-1]

fig, ax = plt.subplots(1, 3, figsize=(10, 4))
ax[0].imshow(image, cmap='gray')
ax[0].set_title('Target Image')
ax[0].set_axis_off()
ax[1].imshow(template, cmap='gray')
ax[1].set_title('Template')
ax[1].set_axis_off()
ax[2].imshow(result, cmap='viridis')
ax[2].set_title('Matching Result (Heatmap)')
ax[2].set_axis_off()
ax[2].plot(x, y, 'r+', markersize=10)
plt.tight_layout()
plt.show()

fig2, ax_main = plt.subplots(figsize=(6, 6))
ax_main.imshow(image, cmap='gray')
ax_main.set_title('Template Found')
ax_main.set_axis_off()
h, w = template.shape
rect = plt.Rectangle((x, y), w, h, edgecolor='r', facecolor='none', lw=2)
ax_main.add_patch(rect)
plt.show()

print(f'Template ditemukan di koordinat (x,y): ({x}, {y})')
print(f'Skor kecocokan maksimum: {result.max():.4f}')

### Hasil Praktikum 4
Area paling terang pada heatmap sesuai dengan lokasi template pada citra asli. Template berhasil ditemukan, tetapi metode ini tidak invariant terhadap rotasi dan perubahan skala.

## Praktikum 5 — Simulasi Content-Based Image Retrieval (CBIR) Sederhana

### Tujuan
Mensimulasikan CBIR sederhana dengan histogram warna dan membandingkannya dengan fitur mean RGB sebagai penugasan tambahan.

In [ ]:
image_db_names = ['astronaut', 'camera', 'coffee', 'coins', 'chelsea']
database_images = []
database_hists = []
database_means = []

for name in image_db_names:
    img = getattr(data, name)()
    if img.ndim == 2:
        img = color.gray2rgb(img)
    img_resized = transform.resize(img, (100, 100), anti_aliasing=True)
    database_images.append(img_resized)
    database_hists.append(calculate_rgb_histogram(img_resized))
    database_means.append(calculate_mean_rgb(img_resized))

query_image_name = 'chelsea'
query_index = image_db_names.index(query_image_name)
query_image = database_images[query_index]
query_hist = database_hists[query_index]
query_mean = database_means[query_index]

hist_distances = [distance.cosine(query_hist, hist) for hist in database_hists]
hist_sorted_indices = np.argsort(hist_distances)
mean_distances = [distance.euclidean(query_mean, mean) for mean in database_means]
mean_sorted_indices = np.argsort(mean_distances)

print('Hasil Retrieval dengan Histogram Warna:')
for rank, idx in enumerate(hist_sorted_indices, start=1):
    print(f'Rank {rank}: {image_db_names[idx]} (Distance: {hist_distances[idx]:.4f})')

print('\nHasil Retrieval dengan Mean RGB:')
for rank, idx in enumerate(mean_sorted_indices, start=1):
    print(f'Rank {rank}: {image_db_names[idx]} (Distance: {mean_distances[idx]:.4f})')

fig, axes = plt.subplots(1, len(database_images) + 1, figsize=(15, 3))
axes[0].imshow(query_image)
axes[0].set_title('Query: chelsea')
axes[0].axis('off')

for i, idx in enumerate(hist_sorted_indices):
    axes[i + 1].imshow(database_images[idx])
    axes[i + 1].set_title(f'{image_db_names[idx]}\nD: {hist_distances[idx]:.3f}')
    axes[i + 1].axis('off')

plt.tight_layout()
plt.show()

### Hasil Praktikum 5
Histogram warna cukup efektif untuk database kecil karena menjaga distribusi warna global. Mean RGB lebih sederhana, tetapi terlalu kasar karena hanya menyimpan nilai rata-rata per kanal.

## F. Penugasan

### 3. Fitur Berbeda untuk CBIR
Pada eksperimen ini, histogram warna memberikan hasil retrieval yang lebih baik daripada mean RGB untuk database kecil tersebut. Histogram lebih kaya informasi karena menyimpan distribusi warna, sedangkan mean RGB hanya menyimpan rata-rata per kanal sehingga kehilangan struktur distribusi warna.

Urutan retrieval histogram yang diperoleh adalah: `chelsea`, `coins`, `coffee`, `astronaut`, `camera`.

Urutan retrieval mean RGB yang diperoleh adalah: `chelsea`, `astronaut`, `coffee`, `camera`, `coins`.

### 4. Template Matching Invariant
Metode `skimage.feature.match_template` standar tidak invariant terhadap rotasi atau perubahan skala. Metode ini paling cocok untuk objek dengan orientasi dan ukuran yang sama seperti template. Jika template diputar atau diperbesar/diperkecil, skor kecocokan biasanya turun.

### Kesimpulan
Jobsheet 5 menunjukkan bahwa pemilihan fitur sangat menentukan hasil pencarian dan pengukuran kemiripan. Fitur sederhana cocok untuk eksperimen dasar, tetapi untuk kasus yang lebih kompleks dibutuhkan fitur yang lebih robust terhadap rotasi, skala, dan variasi pencahayaan.